# Part 2 - Mô hình OLS Baseline cho bài toán Data Fitting

Notebook này xây dựng toàn bộ quy trình mô hình nền tảng **Ordinary Least Squares (OLS)** cho bộ dữ liệu OASIS longitudinal. Mục tiêu dự báo là biến **MMSE**, một thang điểm thường được dùng để đánh giá trạng thái nhận thức.

Quy trình trong notebook tuân thủ các nguyên tắc quan trọng của mô hình hóa dữ liệu:

- Chỉ học các tham số tiền xử lý từ tập huấn luyện.
- Chỉ đánh giá mô hình trên tập kiểm tra được giữ riêng.
- Dùng lại hàm `ols_fit(X, y)` từ Part 1 thông qua lớp `OLSBaseline` có sẵn.
- Kiểm tra kỹ dữ liệu sau tiền xử lý để tránh lỗi lan truyền `NaN` vào mô hình.
- Báo cáo độ chính xác, suy diễn thống kê, VIF và các đồ thị chẩn đoán phần dư.

## 1. Import thư viện

Ta sử dụng `pandas` để xử lý bảng dữ liệu, `numpy` để chuyển dữ liệu sang dạng ma trận số, `matplotlib` để vẽ đồ thị, và `train_test_split` để chia dữ liệu thành tập huấn luyện và tập kiểm tra.

Hai thành phần chính của project được dùng trong notebook này là:

- `DataPipeline`: xử lý missing values, one-hot encoding và chuẩn hóa biến số.
- `OLSBaseline`: huấn luyện, đánh giá, suy diễn hệ số, tính VIF và vẽ diagnostic plots. Bên trong lớp này có gọi lại `ols_fit` từ Part 1.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# Thiết lập đường dẫn project để import được các module part1/part2
# khi chạy notebook từ thư mục part2.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "part2" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from part2.data_pipeline import DataPipeline
from part2.model_comparison import OLSBaseline

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")

## 2. Tái lập kết quả

Trong thực nghiệm thống kê và học máy, một quy trình được xem là tốt khi người khác có thể chạy lại và thu được cùng kết quả. Vì vậy ta cố định `SEED` cho NumPy, bước chia train/test và mô hình OLS baseline.

Điều này đặc biệt quan trọng vì train/test split có yếu tố ngẫu nhiên. Nếu không cố định `random_state`, mỗi lần chạy notebook có thể tạo ra tập kiểm tra khác nhau, dẫn đến metric khác nhau.

In [ ]:
SEED = 42
TEST_SIZE = 0.20

np.random.seed(SEED)

print(f"Seed sử dụng: {SEED}")
print(f"Tỷ lệ test set: {TEST_SIZE:.0%}")

## 3. Đọc dữ liệu

Dữ liệu được đọc từ đường dẫn yêu cầu:

`data/oasis_longitudinal.csv`

Trước khi mô hình hóa, ta quan sát kích thước dữ liệu, vài dòng đầu tiên, kiểu dữ liệu và số lượng giá trị thiếu ở từng cột. Đây là bước kiểm tra nền tảng để hiểu dữ liệu trước khi đưa vào mô hình.

In [ ]:
DATA_PATH = Path("data") / "oasis_longitudinal.csv"

pipeline = DataPipeline()
df_raw = pipeline.load_dataset(str(DATA_PATH))

print(f"Kích thước dữ liệu gốc: {df_raw.shape}")
display(df_raw.head())

data_overview = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "missing_count": df_raw.isna().sum(),
    "missing_rate": df_raw.isna().mean(),
})

display(data_overview)

## 4. Loại bỏ các cột định danh

Hai cột `Subject ID` và `MRI ID` là các biến định danh có cardinality cao. Chúng chủ yếu dùng để nhận diện cá thể hoặc lần chụp MRI, không phải là đặc trưng lâm sàng tổng quát.

Nếu giữ các cột này trong mô hình, mô hình có thể học các mẫu giả liên quan đến mã định danh thay vì học quan hệ thực sự giữa đặc trưng và MMSE. Vì vậy ta loại bỏ chúng trước khi chia dữ liệu.

In [ ]:
ID_COLUMNS = ["Subject ID", "MRI ID"]

df = df_raw.drop(columns=ID_COLUMNS).copy()
df = pipeline.remove_duplicates(df)

print(f"Kích thước sau khi bỏ ID và dòng trùng: {df.shape}")
display(df.head())

## 5. Xử lý giá trị thiếu

Trong bài toán supervised learning, biến mục tiêu `MMSE` bắt buộc phải có giá trị. Nếu một dòng bị thiếu `MMSE`, ta không thể dùng dòng đó để huấn luyện hoặc đánh giá vì không có nhãn thật để so sánh.

Vì vậy notebook chỉ xóa các dòng thiếu target. Các giá trị thiếu ở biến đầu vào sẽ được xử lý bởi `DataPipeline` sau khi chia train/test. Cách làm này tránh rò rỉ dữ liệu vì trung bình, độ lệch chuẩn và mode chỉ được học từ tập huấn luyện.

In [ ]:
TARGET = "MMSE"

rows_before = len(df)
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
rows_after = len(df)

print(f"Số dòng bị xóa do thiếu {TARGET}: {rows_before - rows_after}")

missing_after_target_filter = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_rate": df.isna().mean(),
})

display(missing_after_target_filter)

## 6. Xác định biến đầu vào và biến mục tiêu

Ta đặt:

- `X`: ma trận đặc trưng, gồm tất cả các cột còn lại sau khi bỏ `MMSE`.
- `y`: vector mục tiêu, chính là điểm `MMSE`.

Về mặt lý thuyết OLS, ta giả sử quan hệ tuyến tính:

$$y = X\beta + \varepsilon$$

Trong đó $\beta$ là vector hệ số cần ước lượng và $\varepsilon$ là sai số ngẫu nhiên.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(float)

print(f"Kích thước X: {X.shape}")
print(f"Kích thước y: {y.shape}")
display(X.head())

## 7. Chia tập huấn luyện và tập kiểm tra

Tập huấn luyện dùng để học tham số của pipeline và mô hình. Tập kiểm tra được giữ riêng cho đến bước đánh giá cuối cùng.

Điểm quan trọng: **không fit pipeline trên toàn bộ dữ liệu trước khi chia train/test**. Nếu làm vậy, thống kê từ test set có thể đi vào quá trình chuẩn hóa hoặc imputation, gây data leakage và làm metric lạc quan hơn thực tế.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
)

print(f"X_train_raw: {X_train_raw.shape}")
print(f"X_test_raw:  {X_test_raw.shape}")
print(f"y_train:     {y_train.shape}")
print(f"y_test:      {y_test.shape}")

## 8. Tiền xử lý dữ liệu bằng DataPipeline

`DataPipeline` thực hiện ba nhóm thao tác chính:

1. Impute giá trị thiếu: biến số dùng mean, biến phân loại dùng mode học từ train set.
2. One-hot encode biến phân loại: chuyển các cột dạng chuỗi thành các cột nhị phân.
3. Chuẩn hóa biến số: dùng z-score để đưa biến số về cùng thang đo.

Sau khi pipeline chạy xong, notebook tiếp tục ép toàn bộ dữ liệu về dạng số, thay `inf` bằng `NaN`, rồi dùng median từ training set để điền mọi giá trị không hợp lệ còn sót lại. Đây là lớp bảo vệ bổ sung để tránh lỗi `ValueError: Input contains NaN` khi gọi `evaluate()`.

In [ ]:
pipeline = DataPipeline()

# Fit pipeline chỉ trên tập huấn luyện.
X_train_processed = pipeline.fit_transform(X_train_raw, y_train)

# Transform tập kiểm tra bằng đúng tham số học được từ train set.
X_test_processed = pipeline.transform(X_test_raw)

# Ép toàn bộ dữ liệu về dạng số để phù hợp với OLS tự cài đặt.
X_train_numeric = X_train_processed.apply(pd.to_numeric, errors="coerce")
X_test_numeric = X_test_processed.apply(pd.to_numeric, errors="coerce")

# Bảo đảm train và test có cùng thứ tự cột.
feature_names = X_train_numeric.columns.tolist()
X_test_numeric = X_test_numeric.reindex(columns=feature_names)

# Tính giá trị thay thế từ train set, sau đó dùng cho cả train và test.
train_fill_values = X_train_numeric.replace([np.inf, -np.inf], np.nan).median(numeric_only=True)
train_fill_values = train_fill_values.fillna(0.0)

X_train_numeric = X_train_numeric.replace([np.inf, -np.inf], np.nan).fillna(train_fill_values)
X_test_numeric = X_test_numeric.replace([np.inf, -np.inf], np.nan).fillna(train_fill_values)

# Chuyển sang numpy arrays để truyền vào OLSBaseline.
X_train_array = X_train_numeric.to_numpy(dtype=float)
X_test_array = X_test_numeric.to_numpy(dtype=float)
y_train_array = y_train.to_numpy(dtype=float)
y_test_array = y_test.to_numpy(dtype=float)

print(f"Ma trận train sau tiền xử lý: {X_train_array.shape}")
print(f"Ma trận test sau tiền xử lý:  {X_test_array.shape}")
display(X_train_numeric.head())

## 9. Kiểm tra sanity trước khi huấn luyện

OLS yêu cầu dữ liệu đầu vào là ma trận số hữu hạn. Nếu trong `X_test` hoặc `y_test` còn `NaN`, các hàm metric của scikit-learn sẽ báo lỗi trong lúc đánh giá.

Do đó ta kiểm tra rõ ràng:

- Số lượng `NaN`.
- Số lượng `inf` hoặc `-inf`.
- Kích thước train/test có khớp với target hay không.
- Số cột train và test có giống nhau hay không.

In [ ]:
def summarize_array(name, values):
    """Tóm tắt nhanh trạng thái hữu hạn của một numpy array."""
    values = np.asarray(values, dtype=float)
    return {
        "name": name,
        "shape": values.shape,
        "nan_count": int(np.isnan(values).sum()),
        "inf_count": int(np.isinf(values).sum()),
        "all_finite": bool(np.isfinite(values).all()),
    }

sanity_table = pd.DataFrame([
    summarize_array("X_train_array", X_train_array),
    summarize_array("X_test_array", X_test_array),
    summarize_array("y_train_array", y_train_array),
    summarize_array("y_test_array", y_test_array),
])

display(sanity_table)

assert X_train_array.shape[0] == y_train_array.shape[0]
assert X_test_array.shape[0] == y_test_array.shape[0]
assert X_train_array.shape[1] == X_test_array.shape[1]
assert np.isfinite(X_train_array).all()
assert np.isfinite(X_test_array).all()
assert np.isfinite(y_train_array).all()
assert np.isfinite(y_test_array).all()

print("Tất cả sanity checks đều đạt.")

## 10. Huấn luyện mô hình OLS Baseline

OLS ước lượng hệ số bằng cách tối thiểu hóa tổng bình phương phần dư:

$$\min_{\beta} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Nghiệm ma trận quen thuộc là:

$$\hat{\beta} = (X^TX)^{-1}X^Ty$$

Trong project này, hàm `ols_fit(X, y)` ở Part 1 tự thêm intercept vào ma trận thiết kế. Vì vậy ta truyền vào ma trận đặc trưng **không tự thêm cột intercept**. Lớp `OLSBaseline` gọi lại `ols_fit`, lưu hệ số, phương sai phần dư, fitted values và residuals.

In [ ]:
ols_model = OLSBaseline(random_state=SEED)
ols_model.fit(X_train_array, y_train_array)

print("Huấn luyện OLS baseline thành công.")
print(f"Số hệ số bao gồm intercept: {len(ols_model.beta_hat)}")
print(f"Ước lượng phương sai phần dư sigma^2: {ols_model.sigma2_hat:.4f}")

## 11. Đánh giá mô hình trên held-out test set

Mô hình được đánh giá nghiêm ngặt trên tập kiểm tra chưa từng dùng để fit pipeline hoặc fit OLS. Ba metric được báo cáo là:

- **MAE**: sai số tuyệt đối trung bình, dễ diễn giải theo đơn vị MMSE.
- **RMSE**: căn bậc hai của sai số bình phương trung bình, phạt nặng các sai số lớn.
- **R2_test**: tỷ lệ phương sai của target trên test set được giải thích bởi mô hình.

Vì đã kiểm tra `X_test_array` không chứa `NaN`, bước này sẽ không còn gặp lỗi `ValueError: Input contains NaN`.

In [ ]:
metrics = ols_model.evaluate(X_test_array, y_test_array)

metrics_table = pd.DataFrame([metrics], index=["OLS Baseline"])
display(metrics_table)

## 12. Suy diễn thống kê cho hệ số

Sau khi ước lượng hệ số, ta không chỉ quan tâm mô hình dự báo tốt hay không, mà còn muốn hiểu từng biến có quan hệ như thế nào với `MMSE`.

Bảng suy diễn hệ số gồm:

- `coefficient`: hệ số ước lượng.
- `std_error`: sai số chuẩn của hệ số.
- `t_stat`: thống kê t dùng để kiểm định hệ số khác 0.
- `p_value`: mức ý nghĩa thống kê.
- `ci_lower`, `ci_upper`: khoảng tin cậy 95%.

Nếu `p_value` nhỏ, có bằng chứng thống kê rằng hệ số tương ứng khác 0 trong mô hình tuyến tính đang xét. Tuy nhiên cần diễn giải cẩn thận vì đa cộng tuyến và quan hệ phi tuyến có thể ảnh hưởng đến ý nghĩa của từng hệ số.

In [ ]:
inference_table = ols_model.run_inference(feature_names)

display(inference_table)

## 13. Kiểm tra đa cộng tuyến bằng VIF

Đa cộng tuyến xảy ra khi một biến đầu vào có thể được giải thích khá tốt bởi các biến đầu vào khác. Khi đa cộng tuyến cao, hệ số OLS có thể trở nên kém ổn định: sai số chuẩn lớn hơn, dấu hệ số dễ thay đổi và việc diễn giải từng biến trở nên khó hơn.

Variance Inflation Factor của biến $X_j$ được hiểu là:

$$VIF_j = \frac{1}{1 - R_j^2}$$

Trong đó $R_j^2$ là R-squared khi hồi quy $X_j$ theo các biến còn lại. Quy ước thường dùng: **VIF > 10** là dấu hiệu đa cộng tuyến cao.

In [ ]:
vif_table = ols_model.compute_vif(X_train_array, feature_names)
vif_table = vif_table.sort_values("VIF", ascending=False, na_position="last").reset_index(drop=True)

display(vif_table)

high_vif_table = vif_table[vif_table["High Multicollinearity"]]
print(f"Số biến có VIF > 10: {len(high_vif_table)}")

if len(high_vif_table) > 0:
    display(high_vif_table)

## 14. Chẩn đoán phần dư

Phần dư là sai khác giữa giá trị thật và giá trị mô hình dự báo:

$$e_i = y_i - \hat{y}_i$$

Bốn đồ thị chẩn đoán giúp kiểm tra các giả định và điểm bất thường của OLS:

1. **Residuals vs Fitted**: kiểm tra tính tuyến tính và pattern còn sót lại trong phần dư.
2. **Q-Q Plot**: kiểm tra phần dư có gần phân phối chuẩn hay không.
3. **Scale-Location**: kiểm tra phương sai phần dư có tương đối ổn định hay không.
4. **Cook's Distance**: phát hiện quan sát có ảnh hưởng mạnh đến mô hình.

Các đồ thị này được tạo từ phần dư trên tập huấn luyện vì chúng phản ánh hành vi của mô hình đã fit.

In [ ]:
ols_model.diagnostic_plots()

## 15. Kết luận

Notebook này đã hoàn thành quy trình OLS baseline cho Part 2:

- Đọc đúng bộ dữ liệu OASIS từ `data/oasis_longitudinal.csv`.
- Loại bỏ các cột ID có cardinality cao.
- Xóa an toàn các dòng thiếu target `MMSE`.
- Chia train/test trước khi fit pipeline để tránh data leakage.
- Dùng `DataPipeline.fit_transform` trên training set và `DataPipeline.transform` trên test set.
- Chuyển dữ liệu sau tiền xử lý thành numpy arrays dạng số.
- Kiểm tra và ngăn `NaN` hoặc `inf` lan truyền vào mô hình.
- Huấn luyện OLS baseline bằng lại `ols_fit(X, y)` từ Part 1.
- Đánh giá trên held-out test set với MAE, RMSE và `R2_test`.
- Chạy suy diễn hệ số, phân tích VIF và tạo đủ bốn residual diagnostic plots.

Mô hình OLS đóng vai trò baseline rõ ràng, dễ diễn giải và hữu ích để so sánh với các mô hình phức tạp hơn ở các bước sau.